# Choose the latent-space `tau` for latent-I2SB

Loads the two **frozen** GroupCDL dictionaries (`D_joint`, `D_t1ce`) named in
`config/BraTS/latent_i2sb.json`, visualizes their (real-valued) synthesis filter banks, then
encodes real batches through the **exact training path** (`training.latent_i2sb.encode`) and
histograms the latent codes `z0 = D_t1ce(x0).z` and `z1 = D_joint([x1, cond]).z`.

Why this fixes `tau`: the latent bridge injects `std_sb * N(0, I)` onto **every** code entry
(`q_sample(z0, z1)`), so the spread of these codes is exactly the scale `tau` must match. The
image-domain `0.19` in the config is a placeholder and is wrong in latent units.

**Prereqs:** run on the HPC node with the BraTS data, torch 2.x, and the trained dict checkpoints
(steps 1 & 2). Edit the `os.chdir(...)` path below to your repo root.

In [ ]:
import os, sys, json, yaml
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

os.chdir('/scratch/ee2178/ImMAP')          # <-- EDIT to your repo root (has train.py, config/, trained_nets/)

CONFIG = "config/BraTS/latent_i2sb.json"   # dict paths + data + bridge all live here
SPLIT  = "train"                           # "train" or "val"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load the two frozen dictionaries

Same loader `training/latent_i2sb.py` uses: `load_model` (build + load ckpt + eval), then freeze and (if flex) compile. Dict paths are read straight from the latent-I2SB config so this matches training exactly.

In [ ]:
import datasets                          # side-effect: registers dataset loaders
from datasets.registry import build_loader
from training.common import load_model

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

def load_dict(config_path):
    net = load_model(config_path, device=device)     # build_model + load ckpt + eval()
    for p in net.parameters():
        p.requires_grad_(False)
    net.eval()
    if getattr(net, "attn_backend", None) == "flex":
        net.compile_flex()
    return net

JOINT_CFG = cfg["dicts"]["joint_dict_config"]
T1CE_CFG  = cfg["dicts"]["t1ce_dict_config"]
D_joint = load_dict(JOINT_CFG)
D_t1ce  = load_dict(T1CE_CFG)

print(f"D_joint: C={D_joint.C} M={D_joint.M} sc={D_joint.sc} P={D_joint.P}  <- {JOINT_CFG}")
print(f"D_t1ce : C={D_t1ce.C} M={D_t1ce.M} sc={D_t1ce.sc} P={D_t1ce.P}  <- {T1CE_CFG}")
assert D_joint.M == D_t1ce.M and D_joint.sc == D_t1ce.sc, "dicts must share the latent shape"
M = D_joint.M

## 2. Filter banks (real-valued)

Our dicts are `is_complex=false`, so unlike `visualization/filters.py` (which packs real->R, imag->B and calls `get_B_complex`) each atom is a single real 7x7 kernel, shown with a signed diverging colormap. The **synthesis** dictionary is `D = B[0]` (weight `(M, C, P, P)`): the atoms that map an M-channel code back to the image. `D_t1ce` has C=1 (one panel); `D_joint` has C=4 = `[x_t, FLAIR, T1, T2]` (four panels).

In [ ]:
# tile (N, P, P) kernels into one 2D array with NaN padding
def atom_tiles(W, pad=1):
    W = np.asarray(W, dtype=np.float32)
    N, P, _ = W.shape
    ncol = int(np.ceil(np.sqrt(N)))
    nrow = int(np.ceil(N / ncol))
    Pp = P + pad
    grid = np.full((nrow * Pp + pad, ncol * Pp + pad), np.nan, np.float32)
    for i in range(N):
        r, c = divmod(i, ncol)
        grid[pad + r*Pp: pad + r*Pp + P, pad + c*Pp: pad + c*Pp + P] = W[i]
    return grid

# W: (N, C, P, P) real weights -> one signed-colormap panel per input channel c
def show_atoms(W, title, ch_labels=None, scale_each=False, cmap="RdBu_r"):
    W = W.detach().cpu().float()
    N, C, P, _ = W.shape
    cm = plt.get_cmap(cmap).copy(); cm.set_bad("white")
    fig, axes = plt.subplots(1, C, figsize=(4.2 * C, 4.4))
    axes = np.atleast_1d(axes)
    for c in range(C):
        Wc = W[:, c]
        if scale_each:                                   # per-atom unit-normalize (shape only)
            Wc = Wc / (Wc.abs().amax(dim=(1, 2), keepdim=True) + 1e-8)
        a = float(Wc.abs().max()) + 1e-8
        im = axes[c].imshow(atom_tiles(Wc.numpy()), cmap=cm, vmin=-a, vmax=a)
        lbl = ch_labels[c] if ch_labels else f"ch{c}"
        axes[c].set_title(f"{title}\n{lbl}"); axes[c].set_xticks([]); axes[c].set_yticks([])
        plt.colorbar(im, ax=axes[c], fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

show_atoms(D_t1ce.D.weight, "D_t1ce synthesis (64 atoms)", ch_labels=["T1ce"])

In [ ]:
# joint dict channels follow cat[ch0, cond] with cond_idx=[0,1,3]
show_atoms(D_joint.D.weight, "D_joint synthesis (64 atoms)",
           ch_labels=["x_t (bridge)", "FLAIR", "T1", "T2"])

# analysis (encoder) atoms too, if you want them: A[0] weight is also (M, C, P, P)
# show_atoms(D_t1ce.A[0].weight, "D_t1ce analysis A[0]", ch_labels=["T1ce"])

## 3. Data loader (exact training path)

Built from `cfg["data"][SPLIT]` so `image_key`, `scales`, crop, endpoints and `cond_idx=[0,1,3]` are identical to what latent-I2SB trains on.

In [ ]:
data_cfg = dict(cfg["data"][SPLIT])
data_cfg["num_workers"] = 0
loader = build_loader(data_cfg, shuffle=(SPLIT == "train"), drop_last=(SPLIT == "train"))
print("split:", SPLIT, "| size:", len(loader.dataset), "| batch:", loader.batch_size)
print("image_key:", data_cfg.get("image_key"), "| scales:", data_cfg.get("scales"))
print("x0_idx:", data_cfg.get("x0_idx"), "x1_idx:", data_cfg.get("x1_idx"),
      "cond_idx:", data_cfg.get("cond_idx"))

x0, x1, cond, mask = next(iter(loader))
print("x0", tuple(x0.shape), "| x1", tuple(x1.shape),
      "| cond", tuple(cond.shape), "| mask", tuple(mask.shape))

## 4. Encode to the shared latent

`training.latent_i2sb.encode` is the frozen, `no_grad`, `sigma=0` encode the training loop uses for the bridge endpoints. Both codes must share a shape to bridge.

In [ ]:
from training.latent_i2sb import encode

z0 = encode(D_t1ce, x0.to(device))                              # target latent  (B, M, Q, Q)
z1 = encode(D_joint, torch.cat([x1, cond], dim=1).to(device))   # prior  latent  (B, M, Q, Q)
print("z0 (T1ce latent):", tuple(z0.shape))
print("z1 (joint latent):", tuple(z1.shape))
assert z0.shape == z1.shape, "endpoints must share a shape to bridge"
print("latent spatial", tuple(z0.shape[-2:]), "= image", tuple(x0.shape[-2:]), "/ sc", D_t1ce.sc)

## 5. Latent statistics -- the scale `tau` must match

`tau` is an **absolute** std added to every code entry, so what matters is its size relative to the code spread. `%~0` is the fraction of near-zero entries (GroupCDL codes are group-sparse); when it is high, `std` is pulled down by the zeros, so also watch `std!=0` (std over active entries) and `p99`.

In [ ]:
def qtile(v, ps):
    v = v.flatten().float()
    if v.numel() > 1_000_000:
        v = v[torch.randint(0, v.numel(), (1_000_000,))]
    return torch.quantile(v, torch.as_tensor(ps, dtype=torch.float32))

N_BATCHES = 8
acc = {"z0 (T1ce latent)": [], "z1 (joint latent)": [], "z0 - z1 (latent gap)": []}
it = iter(loader)
for _ in range(N_BATCHES):
    try:
        b0, b1, bc, bm = next(it)
    except StopIteration:
        break
    with torch.no_grad():
        zz0 = encode(D_t1ce, b0.to(device)).flatten().cpu()
        zz1 = encode(D_joint, torch.cat([b1, bc], dim=1).to(device)).flatten().cpu()
    acc["z0 (T1ce latent)"].append(zz0)
    acc["z1 (joint latent)"].append(zz1)
    acc["z0 - z1 (latent gap)"].append(zz0 - zz1)
acc = {k: torch.cat(v).float() for k, v in acc.items()}

print(f"{'quantity':22s} {'mean':>8} {'std':>8} {'std!=0':>8} {'p1':>8} {'p50':>8} {'p99':>8} {'%~0':>6}")
for k, v in acc.items():
    p = qtile(v, [0.01, 0.5, 0.99])
    nz = v[v.abs() > 1e-4]
    std_nz = float(nz.std()) if nz.numel() else 0.0
    frac0 = float((v.abs() <= 1e-4).float().mean()) * 100
    print(f"{k:22s} {v.mean():8.3f} {v.std():8.3f} {std_nz:8.3f} "
          f"{p[0]:8.3f} {p[1]:8.3f} {p[2]:8.3f} {frac0:6.1f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
for a, (k, v) in zip(ax, acc.items()):
    lo, hi = qtile(v, [0.002, 0.998]).tolist()
    a.hist(v.numpy(), bins=200, range=(lo, hi))
    a.set_title(k); a.set_yscale("log"); a.set_xlabel("code value")
plt.tight_layout(); plt.show()

## 6. `tau` ratio guidance

`tau` is scale-dependent, so the useful number is the **ratio** `tau / std(z0)`. The image-domain run used `tau=0.19` at `std(x0_scaled) ~ 0.48`, i.e. `~0.4 x` the signal std (the I2SB paper sits around `0.4-0.6 x`). Aim for a similar fraction of the latent std here.

In [ ]:
CANDIDATE_TAUS = [0.02, 0.05, 0.1, 0.2, 0.4, 0.8]
z0_std  = float(acc["z0 (T1ce latent)"].std())
gap_std = float(acc["z0 - z1 (latent gap)"].std())
print(f"std(z0)      = {z0_std:.4f}   per-element std of the target latent")
print(f"std(z0 - z1) = {gap_std:.4f}   the gap the latent bridge crosses\n")
print(f"{'tau':>8} {'peak std_sb':>12} {'max std_fwd':>12} {'tau/std(z0)':>13} {'tau/std(gap)':>14}")
for tau in CANDIDATE_TAUS:
    print(f"{tau:8.3f} {tau:12.3f} {2*tau:12.3f} {tau/z0_std:13.3f} {tau/gap_std:14.3f}")

R_TARGET = 0.4                                   # match the image-domain / paper ratio
print(f"\nStarting point at ratio {R_TARGET} x std(z0):  tau ~ {R_TARGET*z0_std:.3f}")

## 7. Schedule curves + latent-bridge midpoint check

Left: `std_sb` / `std_fwd` for the chosen `tau` against `std(z0)`. Right: the histogram of a real bridge sample `z_t` at `t=0.5` overlaid on the clean `z0` -- pick the `tau` where the midpoint is clearly noised but the code distribution is not washed out.

In [ ]:
from physics.bbridge import brownian_bridge_schedule
from diffusion.i2sb import q_sample

CHOSEN_TAU = round(R_TARGET * z0_std, 3)
n     = int(cfg["i2sb"]["n_points"])
shape = cfg["i2sb"].get("bridge_shape", "constant")
sched = brownian_bridge_schedule(tau=CHOSEN_TAU, n=n, shape=shape)

t = np.linspace(0, 1, n)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].plot(t, sched.std_sb.numpy(),  label="std_sb (injected latent noise)")
ax[0].plot(t, sched.std_fwd.numpy(), label="std_fwd (R sigma input)")
ax[0].axhline(z0_std, ls=":", c="k", lw=0.8, label="std(z0)")
ax[0].set_xlabel("t  (0 = z0 T1ce latent, 1 = z1 joint latent)"); ax[0].set_ylabel("std"); ax[0].legend()
ax[0].set_title(f"tau={CHOSEN_TAU}: max std_sb={float(sched.std_sb.max()):.3f}, "
                f"max std_fwd={float(sched.std_fwd.max()):.3f}")

k_mid = (n - 1) // 2
lo, hi = qtile(acc["z0 (T1ce latent)"], [0.005, 0.995]).tolist()
ax[1].hist(acc["z0 (T1ce latent)"].numpy(), bins=200, range=(lo, hi), density=True, alpha=0.45, label="z0 (clean)")
for tau in [CHOSEN_TAU, 2 * CHOSEN_TAU]:
    s = brownian_bridge_schedule(tau=tau, n=n, shape=shape)
    torch.manual_seed(0)
    zt = q_sample(s, torch.full((z0.shape[0],), k_mid, dtype=torch.long), z0.cpu(), z1.cpu(), ot_ode=False)
    ax[1].hist(zt.flatten().numpy(), bins=200, range=(lo, hi), density=True, histtype="step",
               label=f"z_t t=0.5, tau={tau:.3f}")
ax[1].set_yscale("log"); ax[1].set_xlabel("code value"); ax[1].legend()
ax[1].set_title("bridge midpoint vs clean latent")
plt.tight_layout(); plt.show()

print(f"When happy, set  i2sb.tau = {CHOSEN_TAU}  in {CONFIG}.")